**Ingest fact data into bronze layer**

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StructType,StructField, StringType, IntegerType, BooleanType, DateType

catalog_name="ecommerce"

take data from raw order items csv files and create table in bronze layer

In [0]:
order_items_schema=StructType([
    StructField("dt",                   StringType(),True),
    StructField("order_ts",             StringType(),True),
    StructField("customer_id",          StringType(),True),
    StructField("order_id",             StringType(),True),
    StructField("item_seq",             StringType(),True),
    StructField("product_id",           StringType(),True),
    StructField("quantity",             StringType(),True),
    StructField("unit_price_currency",  StringType(),True),
    StructField("unit_price",           StringType(),True),
    StructField("discount_pct",         StringType(),True),
    StructField("tax_amount",           StringType(),True),
    StructField("channel",              StringType(),True),
    StructField("coupon_code",          StringType(),True)
])

In [0]:
raw_data_path="/Volumes/ecommerce/source_data/raw/order_items/landing/*.csv"

df=spark.read.option("header","true").option("delimiter",",").schema(order_items_schema).csv(raw_data_path)\
    .withColumn("file_name",F.col("_metadata.file_path"))\
        .withColumn("ingest_timestamp",F.current_timestamp())

display(df.limit(10))

In [0]:
df.count()

In [0]:
df.write.format("delta")\
    .mode("overwrite")\
        .option("mergeSchema","true")\
            .saveAsTable(f"{catalog_name}.bronze.brz_order_items")